In [1]:
import common_imports
from common_imports import *


In [2]:
# preprocess + architecture + weights (as in 3.ipynb — no training / torchinfo).
def preprocess(file_path):
    """Load image from disk → float32 tensor [3, 100, 100], RGB, values in [0, 1]."""
    img_bgr = cv2.imread(str(file_path))
    if img_bgr is None:
        raise ValueError(f"Could not read image: {file_path!r}")
    img_bgr = cv2.resize(img_bgr, (100, 100))
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).permute(2, 0, 1).contiguous().float() / 255.0


class EmbeddingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=10),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True),
            nn.Conv2d(64, 128, kernel_size=7),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True),
            nn.Conv2d(128, 128, kernel_size=4),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True),
            nn.Conv2d(128, 256, kernel_size=4),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(9216, 4096),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.embedding(x)


class L1Dist(nn.Module):
    def forward(self, input_embedding, validation_embedding):
        return torch.abs(input_embedding - validation_embedding)


class SiameseNetwork(nn.Module):
    def __init__(self, embedding_net: nn.Module):
        super().__init__()
        self.embedding = embedding_net
        self.l1 = L1Dist()
        self.classifier = nn.Linear(4096, 1)

    def forward(self, input_img, validation_img):
        inp_e = self.embedding(input_img)
        val_e = self.embedding(validation_img)
        distances = self.l1(inp_e, val_e)
        return torch.sigmoid(self.classifier(distances))


_PART_2_ROOT = os.path.dirname(os.path.dirname(ANC_PATH))
weights_path = os.path.join(_PART_2_ROOT, "siamesemodelv2.pt")
if not os.path.isfile(weights_path):
    raise FileNotFoundError(
        f"Missing weights: {weights_path!r} — train and save in 3.ipynb (torch.save) or copy the .pt into Part_2."
    )

siamese_model = SiameseNetwork(EmbeddingNet())
siamese_model.load_state_dict(torch.load(weights_path, map_location="cpu"))
siamese_model = siamese_model.to(device).eval()
print("Loaded:", weights_path, "| device:", device)


Loaded: /Users/igorhebda/Desktop/HTW-Bio/Part_2/siamesemodelv2.pt | device: mps


In [3]:
# App paths under Part_2 (same root as ANC_PATH in common_imports).
_PART_2_ROOT = os.path.dirname(os.path.dirname(ANC_PATH))
APP_DATA = os.path.join(_PART_2_ROOT, "application_data")
VERIF_DIR = os.path.join(APP_DATA, "verification_images")
INPUT_IMAGE_PATH = os.path.join(APP_DATA, "input_image", "input_image.jpg")

os.makedirs(os.path.dirname(INPUT_IMAGE_PATH), exist_ok=True)
os.makedirs(VERIF_DIR, exist_ok=True)

os.listdir(VERIF_DIR)
INPUT_IMAGE_PATH


def verify(model, detection_threshold, verification_threshold):
    """Compare input_image to each file in verification_images (Siamese, PyTorch)."""
    os.makedirs(VERIF_DIR, exist_ok=True)
    if not os.path.isfile(INPUT_IMAGE_PATH):
        raise FileNotFoundError(f"Missing file: {INPUT_IMAGE_PATH!r}")
    results = []
    names = sorted(f for f in os.listdir(VERIF_DIR) if f.lower().endswith(common_imports.GALLERY_EXTS))
    model.eval()
    input_t = preprocess(INPUT_IMAGE_PATH).unsqueeze(0).to(device)
    with torch.no_grad():
        for image in names:
            val_t = preprocess(os.path.join(VERIF_DIR, image)).unsqueeze(0).to(device)
            out = model(input_t, val_t)
            results.append(float(out.squeeze().cpu()))

    n = len(names)
    arr = np.array(results, dtype=np.float64)
    detection = int(np.sum(arr > detection_threshold))
    verification = detection / max(n, 1)
    verified = verification > verification_threshold
    return results, verified


In [4]:
# --- Center crop + optional mirror (same idea as 2.ipynb) ---
# OpenCV window: click for focus; V = save + verify; Q = quit.

ROI = 250
OFFSET_X = 0
OFFSET_Y = 0
MIRROR_HORIZONTAL = True


def center_square_roi(frame_bgr, side: int, off_x: int = 0, off_y: int = 0):
    """Square crop from the frame center (same as 2.ipynb)."""
    h, w = frame_bgr.shape[:2]
    if h >= side and w >= side:
        x0 = (w - side) // 2 + off_x
        y0 = (h - side) // 2 + off_y
        x0 = max(0, min(x0, w - side))
        y0 = max(0, min(y0, h - side))
        return frame_bgr[y0 : y0 + side, x0 : x0 + side, :]
    s = min(h, w)
    y0 = (h - s) // 2
    x0 = (w - s) // 2
    patch = frame_bgr[y0 : y0 + s, x0 : x0 + s, :]
    return cv2.resize(patch, (side, side))


print(
    "Verification window: click for focus, V = verify, Q = quit. "
    "ROI / OFFSET / MIRROR — same idea as in 2.ipynb."
)

_PART_2_ROOT = os.path.dirname(os.path.dirname(ANC_PATH))
_INPUT_DIR = os.path.join(_PART_2_ROOT, "application_data", "input_image")
os.makedirs(_INPUT_DIR, exist_ok=True)
INPUT_SAVE_PATH = os.path.join(_INPUT_DIR, "input_image.jpg")

print("Verification gallery — add .jpg / .jpeg / .png / .webp files here:")
print(" ", os.path.abspath(VERIF_DIR))

last_results = None
last_verified = None

cap = None
for cam_index in (0, 1, 2, 3, 4):
    c = cv2.VideoCapture(cam_index)
    if c.isOpened():
        ret, test = c.read()
        if ret and test is not None and test.size > 0:
            cap = c
            print(f"Camera: index {cam_index}")
            break
        c.release()

if cap is None or not cap.isOpened():
    print(
        "Could not open the camera (check System Settings → Privacy → Camera, or try another index)."
    )
else:
    # Warm-up reads: first frames are often empty or stale in the buffer (macOS / AVFoundation).
    # We avoid CAP_PROP_BUFFERSIZE; on some Macs it breaks a smooth preview stream.
    for _ in range(30):
        cap.read()

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret or frame is None:
                cv2.waitKey(5)
                continue

            if MIRROR_HORIZONTAL:
                frame = cv2.flip(frame, 1)

            roi = center_square_roi(frame, ROI, OFFSET_X, OFFSET_Y)
            # Draw overlay on a copy only; save a clean ROI (text would hurt verification).
            display = roi.copy()
            cv2.putText(
                display,
                "V=verify Q=quit",
                (4, 18),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                1,
                cv2.LINE_AA,
            )

            cv2.imshow("Verification", display)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("v"):
                cv2.imwrite(INPUT_SAVE_PATH, roi)
                last_results, last_verified = verify(siamese_model, 0.5, 0.5)
                gallery = sorted(f for f in os.listdir(VERIF_DIR) if f.lower().endswith(common_imports.GALLERY_EXTS))
                print("verified:", last_verified, "— preview keeps running; Q = quit.")
                if not gallery:
                    print(
                        "  (no .jpg/.jpeg/.png/.webp in gallery — nothing to compare)"
                    )
                    _all = sorted(os.listdir(VERIF_DIR))
                    print("  Directory:", os.path.abspath(VERIF_DIR))
                    print(
                        "  Contents (all filenames):",
                        _all if _all else "(empty folder)",
                    )
                elif len(gallery) != len(last_results):
                    print("  WARNING: len(gallery)=", len(gallery), "len(scores)=", len(last_results))
                    print("  files:", gallery)
                    print("  scores:", last_results)
                else:
                    _scores = np.asarray(last_results, dtype=np.float64)
                    if _scores.size:
                        print(
                            f"  min = {_scores.min():.6f}, mean = {_scores.mean():.6f}, max = {_scores.max():.6f}"
                        )
            elif key == ord("q"):
                break
    finally:
        cap.release()

cv2.destroyAllWindows()

if last_results is None:
    print(
        "verify() did not run: focus the Verification window, then press V on the keyboard "
        "(not in the Jupyter cell)."
    )
else:
    _arr = np.asarray(last_results, dtype=np.float64)
    print("np.sum(results > 0.9) =", int(np.sum(_arr > 0.9)))
    if _arr.size == 0:
        print(
            "(empty scores — gallery had no .jpg/.jpeg/.png/.webp; "
            "copy images into the directory printed above)"
        )
    last_results


Verification window: click for focus, V = verify, Q = quit. ROI / OFFSET / MIRROR — same idea as in 2.ipynb.
Verification gallery — add .jpg / .jpeg / .png / .webp files here:
  /Users/igorhebda/Desktop/HTW-Bio/Part_2/application_data/verification_images
Camera: index 0
verified: True — preview keeps running; Q = quit.
  min = 0.705755, mean = 0.914537, max = 0.999648
verified: True — preview keeps running; Q = quit.
  min = 0.792427, mean = 0.921407, max = 0.999598
verified: True — preview keeps running; Q = quit.
  min = 0.972347, mean = 0.991904, max = 0.999941
verified: True — preview keeps running; Q = quit.
  min = 0.998333, mean = 0.999367, max = 0.999997
verified: True — preview keeps running; Q = quit.
  min = 0.876309, mean = 0.988049, max = 0.999879
verified: True — preview keeps running; Q = quit.
  min = 0.717343, mean = 0.906045, max = 0.992959
verified: True — preview keeps running; Q = quit.
  min = 0.984125, mean = 0.994073, max = 0.999956
verified: False — preview kee